# HyDE

가상의 정답 문서를 임베딩하면 검색 순위가 달라지는지 확인합니다.

## 개념을 코드에 연결하기

**호출 표기와 실행 시점.** HyDE는 질문에 대한 가상 문서를 먼저 생성하고 그 문장의 임베딩으로 실제 문서를 검색합니다. 가상 문서는 검색 벡터를 만드는 중간 표현이며 근거 문서가 아닙니다. 답변 단계에서는 반드시 검색된 실제 문서의 본문과 출처를 사용합니다.

**다른 상황에 적용.** 전문 용어가 적은 사용자 질문에서 실제 매뉴얼과 비슷한 문체의 가상 안내문을 만들면 관련 문서와의 벡터 거리가 달라질 수 있습니다. 예를 들어 ‘계정 잠겼어’를 잠금 해제 절차 형태의 가상 문장으로 바꿔 검색할 수 있습니다.

**비교와 경계.** 질문 재작성은 질문 형태를 유지하고 HyDE는 답변처럼 보이는 문서를 만듭니다. 가상 문서가 틀린 사실을 포함해도 검색용으로는 작동할 수 있지만, 그 문장을 답변 근거로 사용하면 환각이 섞입니다. 순위 변화는 데이터셋으로 측정해야 합니다.


## 실행 환경 준비

고정한 Python 패키지를 확인합니다.

**이 셀의 코드 읽기.** `%pip install`은 이 노트북 커널이 사용할 패키지를 설치합니다. 앞 셀의 값을 입력으로 사용하지 않으며, 설치가 끝난 뒤의 코드 셀에서 패키지를 가져와 사용합니다.


In [1]:
%pip install -q langchain==1.4.0 langchain-openai==1.6.1 python-dotenv==1.2.3 numpy==2.5.3
import sys
print("준비 완료:", sys.version.split()[0])

Note: you may need to restart the kernel to use updated packages.
준비 완료: 3.12.13


## .env 찾기

가까운 .env에서 키를 읽고 존재 여부만 표시합니다.

**이 셀의 코드 읽기.** `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [2]:
# 현재 폴더에서 가까운 .env를 찾되 비밀 값은 출력하지 않습니다.
import os, time
from dotenv import find_dotenv, load_dotenv
env_path=find_dotenv(usecwd=True)
load_dotenv(env_path)
print("OPENAI_API_KEY:","있음" if os.getenv("OPENAI_API_KEY") else "없음")

OPENAI_API_KEY: 있음


## 공통 데이터

Ch12의 축소 데이터와 Ch11 방식의 정답 문서 ID를 사용합니다.

**이 셀의 코드 읽기.** `json.loads(text)`는 JSON 문자열을 파이썬 딕셔너리나 목록으로 변환해 반환합니다. `read_text()`는 파일 내용을 문자열로 반환합니다. `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [3]:
# 배포 묶음에서도 같은 상대 경로를 찾도록 프로젝트 루트를 거슬러 올라갑니다.
import json
from pathlib import Path
import numpy as np
root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/"labs").is_dir())
data_dir=root/"labs/part4/ch14-rag-patterns/data"
docs=json.loads((data_dir/"documents.json").read_text(encoding="utf-8"))
queries=json.loads((data_dir/"queries.json").read_text(encoding="utf-8"))
print("문서·질문:",len(docs),len(queries))

문서·질문: 12 8


## 모델과 임베딩

모델과 256차원 문서 벡터를 준비합니다.

**이 셀의 코드 읽기.** `OpenAIEmbeddings(...)`는 문자열을 수치 벡터로 변환할 임베딩 클라이언트를 준비합니다. 실제 호출은 `embed_query()`나 `embed_documents()`에서 일어납니다. `embed_documents(texts)`는 여러 문장을 입력받아 문장 수만큼의 벡터를 반환합니다. `init_chat_model(...)`은 채팅 모델 객체를 만들며, 이 줄에서는 아직 답변을 생성하지 않습니다. `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [4]:
# 256차원은 Ch12와 같은 비교 조건이며 행 하나가 문서 하나입니다.
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings
chat=init_chat_model("gpt-4.1-mini",model_provider="openai",temperature=0)
embedder=OpenAIEmbeddings(model="text-embedding-3-small",dimensions=256)
vectors=np.array(embedder.embed_documents([d["text"] for d in docs]))
print("임베딩 배열:",vectors.shape)

임베딩 배열: (12, 256)


## 검색·생성 함수

검색 점수, 토큰, 시간, 추정 비용을 같은 방식으로 기록합니다.

**이 셀의 코드 읽기.** `search(question: str, k: int = 3)`, `ask(prompt: str)`, `show_hits(hits: list[tuple[dict[str, Any], float]])`는 함수를 정의하는 표기입니다. 이 셀에서는 함수 객체만 준비되고 들여쓴 본문은 뒤에서 함수를 호출할 때 실행됩니다. `return` 뒤의 값은 화면 출력이 아니라 호출한 코드로 전달되는 반환값입니다. `invoke(input)`는 정의한 함수를 뒤에서 호출할 때 해당 함수 본문 안에서 모델이나 그래프를 실행하고 응답 객체를 반환합니다. `embed_query(text)`는 검색 질문 한 개를 벡터 하나로 반환합니다.


In [5]:
# 코사인 유사도는 내적을 두 벡터의 길이로 나누어 계산합니다.
from typing import Any
usage_log: list[dict[str, float | int]] = []

def search(question: str, k: int = 3) -> list[tuple[dict[str, Any], float]]:
    q=np.array(embedder.embed_query(question))
    scores=vectors@q/(np.linalg.norm(vectors,axis=1)*np.linalg.norm(q))
    # argsort 결과를 뒤집어 점수가 큰 문서부터 k개를 고릅니다.
    return [(docs[i],float(scores[i])) for i in np.argsort(scores)[::-1][:k]]
def ask(prompt: str) -> tuple[str, int, float]:
    started=time.perf_counter()
    msg=chat.invoke(prompt)
    usage=msg.usage_metadata or {}
    details=usage.get("input_token_details", {})
    input_tokens=usage.get("input_tokens", 0)
    cached_tokens=details.get("cache_read", 0)
    output_tokens=usage.get("output_tokens", 0)
    tokens=usage.get("total_tokens", 0)
    # 2026-09-10 gpt-4.1-mini 공개 단가로 추정하며 실제 청구액과 다를 수 있습니다.
    usd=((input_tokens-cached_tokens)*0.40+cached_tokens*0.10+output_tokens*1.60)/1_000_000
    usage_log.append({"input":input_tokens,"cached":cached_tokens,"output":output_tokens,"usd":usd})
    return msg.text,tokens,round((time.perf_counter()-started)*1000,1)

def show_hits(hits: list[tuple[dict[str, Any], float]]) -> list[tuple[str, float]]:
    return [(d["doc_id"],round(score,3)) for d,score in hits]

## 질문 임베딩 기준선

원 질문을 바로 임베딩해 상위 세 문서를 찾습니다. 정답 ACC-003의 기준선 순위를 기록합니다.

**이 셀의 코드 읽기.** `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [6]:
# 짧은 원 질문의 임베딩으로 정답 문서 순위를 측정합니다.
question="카드를 잃어버렸는데 계정을 보호하려면 무엇을 해야 하나요?"
baseline=search(question,k=3)
print("질문 검색:",show_hits(baseline))

질문 검색: [('ACC-003', 0.42), ('ACC-001', 0.411), ('ACC-004', 0.327)]


## 가상 문서 생성

질문에서 가상 안내문을 만들고 그 문장을 임베딩해 검색합니다. 가상 내용과 실제 문서 순위를 구분해 확인합니다.

**이 셀의 코드 읽기.** `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [7]:
# 가상 안내문은 검색용으로만 생성하고 사실 근거로 사용하지 않습니다.
hypothesis,tokens,latency=ask(f"다음 질문에 답할 법한 도서관 안내문을 2문장으로 가상 작성하세요. 사실 확인용 답이 아닙니다.\n{question}")
hyde=search(hypothesis,k=3)
print("가상 문서:",hypothesis)
print("HyDE 검색:",show_hits(hyde),"토큰:",tokens,"ms:",latency)

가상 문서: 도서관 카드를 분실하셨다면 즉시 안내 데스크에 방문하여 분실 신고를 해주시기 바랍니다. 계정 보호를 위해 비밀번호 변경과 임시 정지 조치를 도와드립니다.
HyDE 검색: [('FAC-004', 0.487), ('ACC-001', 0.456), ('ACC-003', 0.423)] 토큰: 105 ms: 1069.6


## 결과 범위 확인

두 검색에서 정답 ACC-003의 순위를 계산합니다. 순위가 1위에서 3위로 낮아졌는지 확인합니다.

**이 셀의 코드 읽기.** `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [8]:
# 기준선과 HyDE에서 정답 문서의 순위가 바뀌었는지 비교합니다.
gold="ACC-003"
print("기준선 순위:",next((i for i,(d,_) in enumerate(baseline,1) if d["doc_id"]==gold),None))
print("HyDE 순위:",next((i for i,(d,_) in enumerate(hyde,1) if d["doc_id"]==gold),None))

기준선 순위: 1
HyDE 순위: 3
